# 01 · Transcrição de áudio

Transcreve áudio em texto com o Whisper rodando localmente (faster-whisper).

1. Coloque o áudio em `data/` (mp3, m4a, wav, ogg/opus do WhatsApp...)
2. Ajuste a célula de **parâmetros**
3. Rode tudo — as saídas vão para `output/`

A biblioteca está em `src/y_tools/transcricao/`; aqui é só o playground.

In [ ]:
# Importações

import time

import pandas as pd
from dotenv import load_dotenv

from y_tools.logger import setup_logger
from y_tools.paths import DATA_DIR, OUTPUT_DIR
from y_tools.transcricao import (
    LOGGER_NAME,
    MODELOS_DISPONIVEIS,
    listar_audios,
    salvar,
    transcrever_audio,
)

load_dotenv()
logger = setup_logger(LOGGER_NAME)

## Parâmetros

| Parâmetro | Opções |
| --- | --- |
| `MODELO` | `tiny`, `base`, `small` (padrão), `medium`, `turbo`, `large-v3` |
| `IDIOMA` | `"pt"`, `"en"`... ou `None` para detectar |
| `FORMATOS` | `txt`, `srt`, `vtt`, `json` |

In [ ]:
# Parâmetros

ARQUIVO = DATA_DIR / "exemplo.m4a"
MODELO = "small"
IDIOMA = "pt"
FORMATOS = ["txt", "srt", "json"]

print("Áudios disponíveis em data/:")
for audio in listar_audios(DATA_DIR):
    print(f"  {audio.name}")

## Transcrever

* Na primeira vez o modelo é baixado (de ~75 MB no `tiny` a ~3 GB no `large-v3`).
* O modelo fica em cache na memória: rodar de novo a célula não recarrega.

In [ ]:
# Transcrição

resultado = transcrever_audio(ARQUIVO, modelo=MODELO, idioma=IDIOMA)

print(f"Arquivo: {resultado.arquivo.name}")
print(f"Idioma:  {resultado.idioma}")
print(f"Duração: {resultado.duracao:.1f}s  |  Processamento: {resultado.tempo_processamento:.1f}s")
print(f"Velocidade: {resultado.duracao / max(resultado.tempo_processamento, 1e-6):.1f}x tempo real")

In [ ]:
# Texto completo

print(resultado.texto)

In [ ]:
# Segmentos com tempo

df = pd.DataFrame(
    [{"inicio": s.inicio, "fim": s.fim, "duracao": s.fim - s.inicio, "texto": s.texto} for s in resultado.segmentos]
)
df.round(2)

## Salvar

Gera um arquivo por formato em `output/`, com o mesmo nome do áudio.

In [ ]:
# Exportação

for formato in FORMATOS:
    print(salvar(resultado, formato=formato, destino=OUTPUT_DIR))

## Lote

Transcreve todos os áudios de `data/` que ainda não têm `.txt` em `output/`.

In [ ]:
# Transcrição em lote

pendentes = [a for a in listar_audios(DATA_DIR) if not (OUTPUT_DIR / f"{a.stem}.txt").exists()]
print(f"{len(pendentes)} áudio(s) pendente(s)")

inicio = time.perf_counter()
for audio in pendentes:
    r = transcrever_audio(audio, modelo=MODELO, idioma=IDIOMA)
    for formato in FORMATOS:
        salvar(r, formato=formato, destino=OUTPUT_DIR)

print(f"Concluído em {time.perf_counter() - inicio:.1f}s")

## Comparar modelos

? Vale a pena um modelo maior para este áudio? Rode o mesmo trecho em mais de um
modelo e compare tempo e texto. Modelos disponíveis: tiny, base, small, medium, large-v3, turbo.

In [ ]:
# Comparação de modelos

MODELOS_COMPARAR = ["tiny", "small"]

comparacao = []
for nome in MODELOS_COMPARAR:
    r = transcrever_audio(ARQUIVO, modelo=nome, idioma=IDIOMA)
    comparacao.append({"modelo": nome, "segundos": round(r.tempo_processamento, 1), "texto": r.texto[:300]})

pd.DataFrame(comparacao)